##VideoFake | Face Swap / Deep Fake Colab  (English edition without NSFW filter by NeuroModern)
Based on [Facefusion](https://github.com/facefusion/), [roop by s0mad3v](https://github.com/s0md3v/roop) and [dream80 colab](https://github.com/dream80/roop_colab/)  

Please follow me on the twitter: [twitter.com/neuromodern](https://twitter.com/neuromodern)
<a href="https://colab.research.google.com/github/neuromodern/VideoFake/blob/main/VideoFake_colab.ipynb" target="_parent">



<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
#@title 0. Install  NVIDIA CUDA. Enter 2 in output console for confirm
!sudo apt install nvidia-cudnn -y --yes --fix-missing

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
nvidia-cudnn is already the newest version (8.2.4.15~cuda11.4).
0 upgraded, 0 newly installed, 0 to remove and 34 not upgraded.


In [4]:
#@title 1. Install repositories
!git clone https://github.com/neuromodern/VideoFake.git


fatal: destination path 'VideoFake' already exists and is not an empty directory.


In [5]:
#@title 2 Create model folder and set dir
import shutil
import os

directory = "/content/VideoFake/roop/models/"

# Create the directory if it doesn't already exist
if not os.path.exists(directory):
    os.makedirs(directory)
    print(f"Directory '{directory}' created successfully!")
else:
    print(f"Directory '{directory}' already exists!")


%cd /content/VideoFake/roop

Directory '/content/VideoFake/roop/models/' already exists!
/content/VideoFake/roop


In [6]:
#@title 3. Install requirements

!pip install -r requirements.txt

#!pip install onnxruntime-gpu==1.15.0

Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu118
Ignoring tkinterdnd2-universal: markers 'sys_platform == "darwin" and platform_machine == "arm64"' don't match your environment
Ignoring onnxruntime: markers 'python_version != "3.9" and sys_platform == "darwin" and platform_machine != "arm64"' don't match your environment
Ignoring onnxruntime-coreml: markers 'python_version == "3.9" and sys_platform == "darwin" and platform_machine != "arm64"' don't match your environment
Ignoring onnxruntime-silicon: markers 'sys_platform == "darwin" and platform_machine == "arm64"' don't match your environment


In [3]:
#@title 4. Set up video and face sources
source = "/content/ゆきえの顔.png" #@param {type:"string"}
target = "/content/7川村.mov" #@param {type:"string"}
output = "/content/out.mp4" #@param {type:"string"}

In [7]:
!cd /content/VideoFake/roop && \
python run.py --execution-provider cuda \
    --source /content/ゆきえの顔.png \
    -t /content/7川村.mov \
    -o /content/out.mp4 \
    --frame-processor face_swapper face_enhancer \
    --output-video-encode libx264 \
    --output-video-quality 18 \
    --keep-fps \
    --skip-audio \
    --many-faces

Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}, 'CUDAExecutionProvider': {'device_id': '0', 'gpu_mem_limit': '18446744073709551615', 'gpu_external_alloc': '0', 'gpu_external_free': '0', 'gpu_external_empty_cache': '0', 'cudnn_conv_algo_search': 'EXHAUSTIVE', 'cudnn_conv1d_pad_to_nc1d': '0', 'arena_extend_strategy': 'kNextPowerOfTwo', 'do_copy_in_default_stream': '1', 'enable_cuda_graph': '0', 'cudnn_conv_use_max_workspace': '1', 'tunable_op_enable': '0', 'enable_skip_layer_norm_strict_mode': '0', 'tunable_op_tuning_enable': '0'}}
find model: /root/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}, 'CUDAExecutionProvider': {'device_id': '0', 'gpu_mem_limit': '18446744073709551615', 'gpu_external_alloc': '0', 'gpu_external_free': '0', 'gpu_external_empty_cache': '0', 'cudnn_con

In [2]:
!pip uninstall -y jax jaxlib

Found existing installation: jax 0.5.2
Uninstalling jax-0.5.2:
  Successfully uninstalled jax-0.5.2
Found existing installation: jaxlib 0.5.1
Uninstalling jaxlib-0.5.1:
  Successfully uninstalled jaxlib-0.5.1


In [ ]:
# Googleドライブをマウント
from google.colab import drive
import os
from glob import glob

drive.mount('/content/drive')

# 出力フォルダの設定（Googleドライブ）
output_folder = "/content/drive/MyDrive/output_folder"
os.makedirs(output_folder, exist_ok=True)

# ソース画像（.png, .jpg, .jpeg など）を全取得
image_extensions = ['png', 'jpg', 'jpeg']
source_files = []
for ext in image_extensions:
    source_files.extend(glob(f"/content/*.{ext}"))

# ターゲット動画（.mp4, .mov, .webmなど）
video_extensions = ['mp4', 'mov', 'webm', 'mkv']
target_videos = []
for ext in video_extensions:
    target_videos.extend(glob(f"/content/*.{ext}"))

# 作業ディレクトリに移動（roopの実行ディレクトリ）
%cd /content/VideoFake/roop

# すべての画像 × 動画 に対して処理
for source_file in source_files:
    source_name = os.path.splitext(os.path.basename(source_file))[0]

    for target_video in target_videos:
        target_name = os.path.splitext(os.path.basename(target_video))[0]

        # 同名のスキップ（画像と動画が同一名のときなど）
        if os.path.basename(source_file) == os.path.basename(target_video):
            print(f"Skipping same-name file '{source_file}' and '{target_video}'")
            continue

        output_file = f"{output_folder}/{source_name}_{target_name}_swapped.mp4"
        print(f"\n🔁 処理開始: {source_file} -> {target_video}")

        !python run.py \
            --execution-provider cuda \
            --source "{source_file}" \
            -t "{target_video}" \
            -o "{output_file}" \
            --frame-processor face_swapper face_enhancer \
            --output-video-encode libx264 \
            --output-video-quality 18 \
            --keep-fps \
            --skip-audio \
            --many-faces

Mounted at /content/drive
/content/VideoFake/roop

🔁 処理開始: /content/ゆきえの顔.png -> /content/out.mp4
Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}, 'CUDAExecutionProvider': {'device_id': '0', 'gpu_mem_limit': '18446744073709551615', 'gpu_external_alloc': '0', 'gpu_external_free': '0', 'gpu_external_empty_cache': '0', 'cudnn_conv_algo_search': 'EXHAUSTIVE', 'cudnn_conv1d_pad_to_nc1d': '0', 'arena_extend_strategy': 'kNextPowerOfTwo', 'do_copy_in_default_stream': '1', 'enable_cuda_graph': '0', 'cudnn_conv_use_max_workspace': '1', 'tunable_op_enable': '0', 'enable_skip_layer_norm_strict_mode': '0', 'tunable_op_tuning_enable': '0'}}
find model: /root/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}, 'CUDAExecutionProvider': {'device_id': '0', 'gpu_mem_limit': '18446744073709551615'

In [6]:
try:
    # ここに処理を入れる（例：Roopの実行など）
    print(f"Processing '{target_video}'...")
    # 例: run_face_swap(target_video)

except Exception as e:
    print(f"An error occurred: {e}. Skipping '{target_video}' and continuing with the next file.")

NameError: name 'target_video' is not defined

In [2]:
#@title 5. Run
print (source)
Device = "cuda" #@param ["cuda", "cpu"]

Processor = "face_swapper face_enhancer" #@param ["face_swapper face_enhancer", "face_swapper","face_enhancer"]

VideoEncoder = "libx264" #@param ["libx264", "libx265","ibvpx-vp9"]

VideoQuality = "18" #@param {type:"string"}

KeepFPS = True #@param {type:"boolean"}
KeepAudio = True #@param {type:"boolean"}
KeepFrames = True #@param {type:"boolean"}
ManyFaces = True #@param {type:"boolean"}

KeepFPS ="--keep-fps" if KeepFPS==True else ""
KeepAudio ="--skip-audio" if KeepAudio==True else ""
KeepFrames ="--keep-frames" if KeepFrames==True else ""
ManyFaces ="--many-faces" if ManyFaces==True else ""

#new
#cmd = f"run.py --execution-provider {Device} --source {source} -t {target} -o {output} --frame-processor {Processor} --video-encoder {VideoEncoder} --video-quality {VideoQuality} {KeepFPS} {KeepAudio} {KeepFrames} {ManyFaces}"

#old
cmd = f"run.py --execution-provider {Device} --source {source} -t {target} -o {output} --frame-processor {Processor} --output-video-encode {VideoEncoder} --output-video-quality {VideoQuality} {KeepFPS} {KeepAudio} {KeepFrames} {ManyFaces}"
print("cmd:"+cmd)
!python $cmd


NameError: name 'source' is not defined